In [235]:
import jax
import jax.numpy as jnp

from sklearn.datasets import make_s_curve
import sklearn

import matplotlib.pyplot as plt
import plotly.express as px

from jax.example_libraries import stax

import numpyro
import numpyro.distributions as dist
from numpyro.infer import SVI, Trace_ELBO
from numpyro import optim
from numpyro.infer import Predictive
from numpyro import handlers

from morbius import make_mobius_strip

In [215]:
def plot3d(d, title):
    fig = px.scatter_3d(x=d[:,0], y=d[:,1], z=d[:,2], title=title)
    fig.update_traces(marker=dict(size=2))


    fig.update_layout(
        title_text="S shape",
        title_x=0.5,              
        title_y=0.95,               
        title_font_family="Arial",  
        title_font_size=24,        
        title_font_color="navy",     
        margin=dict(t=60)            
    )

    fig.show()


In [ ]:
import numpy as np

n_samples = 2000

X_mobius, _  = make_mobius_strip(n_samples=2000, radius = 2, width = 1)

plot3d(X_mobius, "Möbius Strip")

In [225]:
X_S, t = make_s_curve(noise=0.10, random_state=0, n_samples=1000)
plot3d(X_S, "Curve Data")

In [226]:
X = X_mobius

In [227]:

def encoder(hidden_dim, z_dim):
    return stax.serial(
        stax.Dense(hidden_dim, W_init=stax.randn()),
        stax.Softplus,
        stax.FanOut(2),
        stax.parallel(
            stax.Dense(z_dim, W_init=stax.randn()),
            stax.serial(stax.Dense(z_dim, W_init=stax.randn()), stax.Exp),
        ),
    )


def decoder(hidden_dim, out_dim):
    return stax.serial(
        stax.Dense(hidden_dim, W_init=stax.randn()),
        stax.Softplus,
        stax.Dense(out_dim, W_init=stax.randn())
    )


def model(batch, hidden_dim=5, z_dim=2, inference=False, **kwargs):
    batch = jnp.reshape(batch, (batch.shape[0], -1))
    batch_dim, out_dim = jnp.shape(batch)
    
    decode = numpyro.module("decoder", decoder(hidden_dim, out_dim), (batch_dim, z_dim))
    
    with numpyro.plate("batch", batch_dim):

        z = numpyro.sample("z", dist.Normal(0, 1).expand([z_dim]).to_event(1))
        
        img_loc = decode(z)
        
        numpyro.deterministic("clean_reconstruction", img_loc)
        
        return numpyro.sample(
            "obs", 
            dist.Normal(img_loc, scale=0.1).to_event(1), 
            obs=(None if inference else batch)
        )


def guide(batch, hidden_dim=5, z_dim=2, **kwargs):
    batch = jnp.reshape(batch, (batch.shape[0], -1))
    batch_dim, out_dim = jnp.shape(batch)
    encode = numpyro.module("encoder", encoder(hidden_dim, z_dim), (batch_dim, out_dim))
    z_loc, z_std = encode(batch)
    with numpyro.plate("batch", batch_dim):
        return numpyro.sample("z", dist.Normal(z_loc, z_std).to_event(1))


In [232]:
X = jnp.array(X)

lr=5.0e-3
hidden_dim = 5
z_dim = 2

adam = optim.Adam(lr)

svi = SVI(model,guide,adam, Trace_ELBO())

In [233]:
rng_key, _ = jax.random.split(rng_key)
rng_key = jax.random.key(42)

svi_result = svi.run(rng_key, 1000, batch=X, hidden_dim=hidden_dim, z_dim = z_dim)



100%|██████████| 1000/1000 [00:05<00:00, 184.39it/s, init loss: 100454.4766, avg. loss [951-1000]: 24909.6523]


In [234]:
rng_key, _ = jax.random.split(rng_key)


predictive = Predictive(
    model, 
    guide=guide, 
    params=svi_result.params, 
    num_samples=1 
)

predictions = predictive(rng_key, batch=X, hidden_dim=hidden_dim, z_dim=z_dim, inference=True)

reconstructed = predictions["clean_reconstruction"][0]

plot3d(reconstructed, "Reconstructed")

In [231]:
sklearn.metrics.mean_squared_error(X.reshape(-1), reconstructed.reshape(-1))

0.18867917358875275